# DentalOS on-device X-ray models — training (Kaggle / Colab GPU)

Trains the two models that run inside the DentalOS phone app:
1. **pathology** — YOLOv8s, 14 classes (caries, periapical lesion, bone loss, impacted tooth, fillings, crowns, RCT, implants…) at 960 px
2. **teeth** — YOLOv8n FDI tooth numbering (32 teeth) at 640 px

Data (public, correctly labelled): `mostafabehroozi/panoramic_radiography_yolo_dataset_14_classes` and DENTEX (`tirandazdylan/dentex-panoramic`, CC BY 4.0).

**Why the old notebook failed:** it merged datasets without remapping class ids and looked for labels next to the images, so the model trained on (almost) no labels and output ~1% confidence on real films.

Outputs in `/kaggle/working/models`: `pathology.onnx/.json`, `teeth.onnx/.json` → copy into `app/public/models/` and rebuild, or import them in the app (Settings → Import improved model).

Enable **GPU** and **Internet** in the notebook settings.

In [ ]:
!pip install -q ultralytics onnx onnxruntime onnxslim pyarrow huggingface_hub opencv-python-headless

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile, os
RAW='/kaggle/working/raw'; os.makedirs(RAW+'/dentex', exist_ok=True)
z = hf_hub_download('mostafabehroozi/panoramic_radiography_yolo_dataset_14_classes', 'panoramic_radiography_yolo_dataset_14_classes.zip', repo_type='dataset')
zipfile.ZipFile(z).extractall(RAW+'/pano14')
for cfg in ['diagnosis','enumeration']:
    os.makedirs(f'{RAW}/dentex/{cfg}', exist_ok=True)
    for split in ['train','validation','test']:
        p = hf_hub_download('tirandazdylan/dentex-panoramic', f'{cfg}/{split}-00000-of-00001.parquet', repo_type='dataset')
        os.system(f'cp {p} {RAW}/dentex/{cfg}/{split}.parquet')
print(os.listdir(RAW+'/pano14'))

In [ ]:
%%writefile prepare_data.py
"""
Prepare clean, correctly-labelled YOLO datasets for DentalOS.

Why this exists: the original Kaggle notebook merged four datasets without remapping
their class IDs and looked for label files next to the images (YOLO keeps them in a
parallel labels/ folder), so the shipped model learned almost nothing (max confidence
~0.01 on real panoramic films). This script builds two datasets with verified labels:

1. pathology/  - 14-class panoramic pathology + restoration dataset (YOLO format,
                 fully annotated). Source: HF `mostafabehroozi/panoramic_radiography_yolo_dataset_14_classes`.
2. teeth/      - 32-class FDI tooth-numbering dataset built from DENTEX enumeration
                 (CC BY 4.0). Source: HF `tirandazdylan/dentex-panoramic`.
   Plus dentex_dx_test/ - DENTEX diagnosis test split, used as an *external* check
                 for caries / periapical lesion / impacted tooth.

Usage:
    python training/prepare_data.py --raw D:/DentalOS-train/raw --out D:/DentalOS-train/data
"""
from __future__ import annotations

import argparse
import io
import shutil
from pathlib import Path

# Final DentalOS pathology taxonomy (index == class id in the 14-class source dataset).
PATHOLOGY_CLASSES = [
    "implant",               # 0 Implant
    "crown_bridge",          # 1 Prosthetic restoration
    "filling",               # 2 Obturation
    "root_canal_treatment",  # 3 Endodontic treatment
    "caries",                # 4 Carious lesion
    "bone_loss",             # 5 Bone resorption
    "impacted_tooth",        # 6 Impacted tooth
    "periapical_lesion",     # 7 Apical periodontitis
    "root_fragment",         # 8 Root fragment
    "furcation_lesion",      # 9 Furcation lesion
    "apical_surgery",        # 10 Apical surgery
    "root_resorption",       # 11 Root resorption
    "orthodontic_device",    # 12 Orthodontic device
    "surgical_device",       # 13 Surgical device
]

# FDI numbering: class id i -> tooth number
FDI_TEETH = [q * 10 + t for q in (1, 2, 3, 4) for t in range(1, 9)]

# DENTEX diagnosis ids -> DentalOS pathology ids
DENTEX_DX_TO_PATHOLOGY = {0: 6, 1: 4, 2: 7, 3: 4}  # Impacted, Caries, Periapical, Deep caries


def write_yaml(path: Path, names: list[str], train: str, val: str, test: str | None = None) -> None:
    lines = [f"path: {path.parent.as_posix()}", f"train: {train}", f"val: {val}"]
    if test:
        lines.append(f"test: {test}")
    lines.append("names:")
    lines += [f"  {i}: {n}" for i, n in enumerate(names)]
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def prepare_pathology(raw: Path, out: Path) -> None:
    src = raw / "pano14"
    dst = out / "pathology"
    splits = {
        "train": src / "train",
        "val": src / "valid",
        "test": src / "test",
        "ext": src / "test_alte_cabinete" / "Ext-validation",
    }
    for name, s in splits.items():
        for kind in ("images", "labels"):
            (dst / kind / name).mkdir(parents=True, exist_ok=True)
        n = 0
        for img in sorted((s / "images").iterdir()):
            lbl = s / "labels" / (img.stem + ".txt")
            if not lbl.exists():
                continue
            rows = []
            for line in lbl.read_text().splitlines():
                parts = line.split()
                if len(parts) != 5:
                    continue
                cls = int(parts[0])
                if not 0 <= cls < len(PATHOLOGY_CLASSES):
                    raise ValueError(f"Unexpected class {cls} in {lbl}")
                rows.append(line.strip())
            shutil.copy2(img, dst / "images" / name / img.name)
            (dst / "labels" / name / (img.stem + ".txt")).write_text("\n".join(rows) + "\n")
            n += 1
        print(f"pathology/{name}: {n} images")
    write_yaml(dst / "data.yaml", PATHOLOGY_CLASSES, "images/train", "images/val", "images/test")
    write_yaml(dst / "data_ext.yaml", PATHOLOGY_CLASSES, "images/train", "images/ext")


def _parquet_rows(path: Path):
    import pyarrow.parquet as pq

    table = pq.read_table(path)
    return table.to_pylist()


def prepare_teeth(raw: Path, out: Path) -> None:
    from PIL import Image

    dst = out / "teeth"
    for split, fname in (("train", "train"), ("val", "validation"), ("test", "test")):
        (dst / "images" / split).mkdir(parents=True, exist_ok=True)
        (dst / "labels" / split).mkdir(parents=True, exist_ok=True)
        rows = _parquet_rows(raw / "dentex" / "enumeration" / f"{fname}.parquet")
        for r in rows:
            img_bytes = r["image"]["bytes"]
            im = Image.open(io.BytesIO(img_bytes))
            w, h = im.size
            name = f"{r['film_id']}"
            (dst / "images" / split / f"{name}.jpg").write_bytes(img_bytes)
            objs = r["objects"]
            lines = []
            for bbox, q, t in zip(objs["bbox"], objs["quadrant"], objs["tooth"]):
                fdi = q * 10 + t
                if fdi not in FDI_TEETH:
                    continue
                cls = FDI_TEETH.index(fdi)
                x, y, bw, bh = bbox
                lines.append(f"{cls} {(x + bw / 2) / w:.6f} {(y + bh / 2) / h:.6f} {bw / w:.6f} {bh / h:.6f}")
            (dst / "labels" / split / f"{name}.txt").write_text("\n".join(lines) + "\n")
        print(f"teeth/{split}: {len(rows)} images")
    write_yaml(dst / "data.yaml", [str(t) for t in FDI_TEETH], "images/train", "images/val", "images/test")


def prepare_dentex_dx_test(raw: Path, out: Path) -> None:
    """DENTEX diagnosis test split remapped to DentalOS ids (only caries/periapical/impacted labelled)."""
    from PIL import Image

    dst = out / "dentex_dx_test"
    (dst / "images").mkdir(parents=True, exist_ok=True)
    (dst / "labels").mkdir(parents=True, exist_ok=True)
    rows = _parquet_rows(raw / "dentex" / "diagnosis" / "test.parquet")
    for r in rows:
        img_bytes = r["image"]["bytes"]
        w, h = Image.open(io.BytesIO(img_bytes)).size
        (dst / "images" / f"{r['film_id']}.jpg").write_bytes(img_bytes)
        objs = r["objects"]
        lines = []
        for bbox, dx in zip(objs["bbox"], objs["diagnosis"]):
            x, y, bw, bh = bbox
            cls = DENTEX_DX_TO_PATHOLOGY[int(dx)]
            lines.append(f"{cls} {(x + bw / 2) / w:.6f} {(y + bh / 2) / h:.6f} {bw / w:.6f} {bh / h:.6f}")
        (dst / "labels" / f"{r['film_id']}.txt").write_text("\n".join(lines) + "\n")
    print(f"dentex_dx_test: {len(rows)} images")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--raw", type=Path, required=True)
    ap.add_argument("--out", type=Path, required=True)
    a = ap.parse_args()
    prepare_pathology(a.raw, a.out)
    prepare_teeth(a.raw, a.out)
    prepare_dentex_dx_test(a.raw, a.out)


In [ ]:
!python prepare_data.py --raw /kaggle/working/raw --out /kaggle/working/data

In [ ]:
%%writefile compress_onnx.py
"""
Halve an ONNX model's download size without changing its maths at runtime:
float32 weights are stored as float16 and cast back to float32 when the session loads
(compute stays fp32, so it runs on every phone's WASM backend). Verifies the
compressed model against the original on random input and sample images.

    python training/compress_onnx.py in.onnx out.onnx [--check-images DIR]
"""
from __future__ import annotations

import argparse
import glob
import os

import numpy as np
import onnx
from onnx import helper, numpy_helper, TensorProto


def compress(src: str, dst: str, min_elems: int = 1024) -> None:
    m = onnx.load(src)
    g = m.graph
    new_inits, casts = [], []
    for init in g.initializer:
        if init.data_type == TensorProto.FLOAT and np.prod(init.dims) >= min_elems:
            arr = numpy_helper.to_array(init).astype(np.float16)
            h = numpy_helper.from_array(arr, init.name + "__fp16")
            new_inits.append(h)
            casts.append(helper.make_node("Cast", [h.name], [init.name], to=TensorProto.FLOAT, name=init.name + "__cast"))
        else:
            new_inits.append(init)
    del g.initializer[:]
    g.initializer.extend(new_inits)
    nodes = list(g.node)
    del g.node[:]
    g.node.extend(casts + nodes)
    onnx.checker.check_model(m)
    onnx.save(m, dst)
    print(f"{src}: {os.path.getsize(src)/1e6:.1f} MB -> {dst}: {os.path.getsize(dst)/1e6:.1f} MB")


def letterbox(img: np.ndarray, size: int) -> np.ndarray:
    import cv2

    h, w = img.shape[:2]
    r = min(size / h, size / w)
    nh, nw = round(h * r), round(w * r)
    H, W = -(-nh // 32) * 32, -(-nw // 32) * 32
    out = np.full((H, W, 3), 114, np.uint8)
    t, l = (H - nh) // 2, (W - nw) // 2
    out[t : t + nh, l : l + nw] = cv2.resize(img, (nw, nh))
    return out.transpose(2, 0, 1)[None].astype(np.float32) / 255.0


def verify(a: str, b: str, images: str | None, imgsz: int) -> None:
    import onnxruntime as ort

    sa, sb = ort.InferenceSession(a), ort.InferenceSession(b)
    name = sa.get_inputs()[0].name
    inputs = [np.random.rand(1, 3, imgsz, imgsz).astype(np.float32)]
    if images:
        import cv2

        for f in sorted(glob.glob(os.path.join(images, "*.jpg")))[:4]:
            inputs.append(letterbox(cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB), imgsz))
    for x in inputs:
        oa = sa.run(None, {name: x})[0]
        ob = sb.run(None, {name: x})[0]
        sc_a, sc_b = oa[0, 4:], ob[0, 4:]
        print(f"shape {x.shape}: max_dbox={np.abs(oa[0,:4]-ob[0,:4]).max():.3f}px  max_dscore={np.abs(sc_a-sc_b).max():.4f}  top score {sc_a.max():.3f} vs {sc_b.max():.3f}")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("src")
    ap.add_argument("dst")
    ap.add_argument("--check-images")
    ap.add_argument("--imgsz", type=int, default=960)
    a = ap.parse_args()
    compress(a.src, a.dst)
    verify(a.src, a.dst, a.check_images, a.imgsz)


In [ ]:
%%writefile train.py
"""
Train the two DentalOS on-device models and export them for the phone app.

    python training/train.py pathology --data D:/DentalOS-train/data/pathology/data.yaml
    python training/train.py teeth     --data D:/DentalOS-train/data/teeth/data.yaml

Defaults are sized for a 4 GB laptop GPU (RTX 3050). On Kaggle/Colab (T4/P100) raise
--batch to 16 and keep everything else. Fine-tuning from dentist feedback: pass the
dataset exported from the app (Settings -> AI learning -> Export training data) as
--data and --weights pointing at the previous best.pt (see training/README.md).
"""
from __future__ import annotations

import argparse
import json
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent))

PRESETS = {
    # Pathology: small lesions on large panoramic films -> 960 px, YOLOv8s (good accuracy/speed on phones).
    "pathology": dict(model="yolov8s.pt", imgsz=960, epochs=50, batch=6, fliplr=0.5),
    # Teeth numbering: left/right matters for FDI numbers -> NO horizontal flip. Teeth are large -> 640 + nano.
    "teeth": dict(model="yolov8n.pt", imgsz=640, epochs=60, batch=16, fliplr=0.0),
}


def keep_awake() -> None:
    """Stop Windows from sleeping while training runs (released automatically when the process exits)."""
    if sys.platform == "win32":
        import ctypes

        ES_CONTINUOUS, ES_SYSTEM_REQUIRED = 0x80000000, 0x00000001
        ctypes.windll.kernel32.SetThreadExecutionState(ES_CONTINUOUS | ES_SYSTEM_REQUIRED)


def main() -> None:
    keep_awake()
    ap = argparse.ArgumentParser()
    ap.add_argument("task", choices=PRESETS)
    ap.add_argument("--data", required=True)
    ap.add_argument("--weights", help="start from these weights (fine-tuning)")
    ap.add_argument("--epochs", type=int)
    ap.add_argument("--batch", type=int)
    ap.add_argument("--imgsz", type=int)
    ap.add_argument("--project", default="D:/DentalOS-train/runs")
    ap.add_argument("--out", default=str(Path(__file__).resolve().parents[1] / "app" / "public" / "models"))
    ap.add_argument("--workers", type=int, default=8)
    ap.add_argument("--cache", default="disk", help="disk | ram | none (disk caches resized images; much faster)")
    ap.add_argument("--resume", action="store_true", help="continue an interrupted run from its last.pt")
    a = ap.parse_args()

    from ultralytics import YOLO

    p = dict(PRESETS[a.task])
    for k in ("epochs", "batch", "imgsz"):
        if getattr(a, k):
            p[k] = getattr(a, k)
    last = Path(a.project) / a.task / "weights" / "last.pt"
    if a.resume and last.exists():
        print(f"Resuming from {last}")
        YOLO(str(last)).train(resume=True)
    else:
        _train(YOLO(a.weights or p["model"]), a, p)
    _export(a, p)


def _train(model, a, p) -> None:
    model.train(
        data=a.data,
        epochs=p["epochs"],
        imgsz=p["imgsz"],
        batch=p["batch"],
        fliplr=p["fliplr"],
        workers=a.workers,
        cache=False if a.cache == "none" else a.cache,
        project=a.project,
        name=a.task,
        exist_ok=True,
        patience=25,
        cos_lr=True,
        close_mosaic=10,
        amp=True,
        plots=True,
        seed=0,
    )


def _export(a, p) -> None:
    from ultralytics import YOLO

    best = Path(a.project) / a.task / "weights" / "best.pt"
    trained = YOLO(str(best))
    metrics = trained.val(data=a.data, imgsz=p["imgsz"], split="test", batch=p["batch"], plots=False)
    onnx_path = Path(trained.export(format="onnx", imgsz=p["imgsz"], dynamic=True, simplify=True, opset=17))

    out_dir = Path(a.out)
    out_dir.mkdir(parents=True, exist_ok=True)
    # store weights as fp16 (half the download, identical fp32 maths at runtime)
    from compress_onnx import compress

    compress(str(onnx_path), str(out_dir / f"{a.task}.onnx"))
    names = trained.names
    meta = {
        "task": a.task,
        "version": f"{a.task}-{__import__('datetime').date.today().isoformat()}",
        "imgsz": p["imgsz"],
        "classes": [names[i] for i in sorted(names)],
        "test_map50": round(float(metrics.box.map50), 4),
        "test_map50_95": round(float(metrics.box.map), 4),
        "per_class_map50": {names[i]: round(float(v), 4) for i, v in zip(metrics.box.ap_class_index, metrics.box.ap50)},
    }
    (out_dir / f"{a.task}.json").write_text(json.dumps(meta, indent=2))
    print(json.dumps(meta, indent=2))


if __name__ == "__main__":
    main()


## Train (≈1–2 h on a T4)

In [ ]:
!python train.py pathology --data /kaggle/working/data/pathology/data.yaml --project /kaggle/working/runs --out /kaggle/working/models --batch 16 --workers 4 --cache ram

In [ ]:
!python train.py teeth --data /kaggle/working/data/teeth/data.yaml --project /kaggle/working/runs --out /kaggle/working/models --batch 32 --workers 4 --cache ram

## External check on DENTEX (caries / periapical / impacted)
The DENTEX test films were never seen in training — this is the honest estimate for those three findings.

In [ ]:
from ultralytics import YOLO
import yaml
names = YOLO('/kaggle/working/runs/pathology/weights/best.pt').names
yaml.safe_dump({'path':'/kaggle/working/data/dentex_dx_test','train':'images','val':'images','names':names}, open('dentex.yaml','w'))
m = YOLO('/kaggle/working/runs/pathology/weights/best.pt').val(data='dentex.yaml', imgsz=960, batch=8, classes=[4,6,7])
print({names[i]: round(float(v),3) for i, v in zip(m.box.ap_class_index, m.box.ap50)})

## Fine-tune with dentist-reviewed data from the app (human-in-the-loop)
In the app: More → AI accuracy & learning → Export training data. Upload the zip as a Kaggle dataset, then:

In [ ]:
# !unzip -q /kaggle/input/<your-export>/dentalos_training_*.zip -d /kaggle/working/feedback
# Mix with the original data so the model does not forget:
# !python train.py pathology --data /kaggle/working/feedback/data.yaml --weights /kaggle/working/runs/pathology/weights/best.pt --epochs 30 --batch 16 --project /kaggle/working/runs_ft --out /kaggle/working/models_ft